<a href="https://colab.research.google.com/github/marcouras/AI-engineering-fundamentals/blob/main/lezione4/Lezione4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# 🤖 AI Engineering Fundamentals
## Lezione 4 — RAG: Conoscenza Personalizzata

**ITS Novitas 4.0 — Sviluppatore Intelligenza Artificiale**  
Docente: Marco Uras | 📅 Giovedì 28/05/2026

---

### 🎯 Obiettivi
- ✅ Capire la pipeline RAG completa
- ✅ Indicizzare un PDF con ChromaDB
- ✅ Implementare la ricerca semantica
- ✅ Integrare RAG nel chatbot esistente

In [1]:
# Setup
!pip install anthropic chromadb pypdf sentence-transformers -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(domanda, system=None, max_tokens=800):
    params = {"model":"claude-haiku-4-5-20251001","max_tokens":max_tokens,
              "messages":[{"role":"user","content":domanda}]}
    if system: params["system"] = system
    return client.messages.create(**params).content[0].text

print("✅ Setup completato!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

---
## 1. Crea un documento di test

Per l'esercizio creiamo un documento di testo su WiData. In un progetto reale useresti un PDF vero.

In [2]:
# Creiamo un documento di testo su WiData
documento_widata = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""

# Salva su file
with open("manuale_widata.txt", "w", encoding="utf-8") as f:
    f.write(documento_widata)

print(f"✅ Documento creato: {len(documento_widata)} caratteri")

✅ Documento creato: 1846 caratteri


---
## 2. Chunking e Indicizzazione

In [3]:
def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = chunka_testo(documento_widata)
print(f"📊 Numero di chunk: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} char) ---")
    print(chunk[:100]+"...")
    print()

📊 Numero di chunk: 6

--- Chunk 1 (400 char) ---

WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è proge...

--- Chunk 2 (400 char) ---
. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n...

--- Chunk 3 (400 char) ---
km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G ...

--- Chunk 4 (400 char) ---
iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-tim...

--- Chunk 5 (400 char) ---
va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiest...

--- Chunk 6 (96 char) ---
: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...



In [4]:
import chromadb

# Crea il client ChromaDB in memoria
chroma_client = chromadb.Client()

# Crea o recupera la collection
collection = chroma_client.get_or_create_collection(
    name="widata_docs",
    metadata={"hnsw:space": "cosine"}  # usa similarità coseno
)

# Indicizza i chunk
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Indicizzati {collection.count()} chunk in ChromaDB")
print("💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 40.2MiB/s]


✅ Indicizzati 6 chunk in ChromaDB
💡 ChromaDB ha calcolato automaticamente gli embedding per ogni chunk!


---
## 3. Ricerca Semantica

In [5]:
def cerca(domanda, n_risultati=3):
    """Cerca i chunk più rilevanti per la domanda."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

# Test ricerca semantica
domande_test = [
    "Quali sensori supportate per ambienti esterni?",
    "Come posso integrare i dati con il mio sistema ERP?",
    "Qual è il costo del piano professionale?",
]

for domanda in domande_test:
    print(f"\n❓ {domanda}")
    chunks_trovati = cerca(domanda, n_risultati=2)
    for i, chunk in enumerate(chunks_trovati):
        print(f"  📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali sensori supportate per ambienti esterni?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Come posso integrare i dati con il mio sistema ERP?
  📄 Chunk 1: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...
  📄 Chunk 2: iData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino...

❓ Qual è il costo del piano professionale?
  📄 Chunk 1: : Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
...
  📄 Chunk 2: va.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptim...


---
## 4. RAG Completo — Domanda + Contesto + Risposta

In [6]:
SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, azienda IoT e smart cities di Sassari.
Rispondi SOLO basandoti sui documenti forniti nel contesto.
Se la risposta non è nei documenti, dì chiaramente: 'Non ho questa informazione nei miei documenti.'
Non inventare mai informazioni. Sii conciso e preciso.
"""

def chat_rag(domanda, n_chunks=3):
    """Chatbot con RAG: recupera contesto e genera risposta."""
    # 1. Recupera i chunk rilevanti
    chunks_rilevanti = cerca(domanda, n_risultati=n_chunks)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    # 2. Costruisci il prompt aumentato
    prompt = f"""Documenti di riferimento:

{contesto}

---

Domanda dell'utente: {domanda}"""

    # 3. Genera la risposta
    risposta = chiedi_claude(prompt, system=SYSTEM_WIDATA)
    return risposta, chunks_rilevanti

# Test completo
domanda = "Il sensore XS200 funziona in ambienti molto freddi?"
risposta, chunks = chat_rag(domanda)

print(f"❓ {domanda}")
print(f"\n🤖 {risposta}")
print(f"\n📄 Basato su {len(chunks)} chunk")

❓ Il sensore XS200 funziona in ambienti molto freddi?

🤖 Sì, il sensore XS200 può funzionare in ambienti freddi. Secondo le specifiche tecniche, misura temperature da **-20°C a +60°C**, quindi è idoneo per ambienti molto freddi fino a -20°C.

Inoltre, è classificato **IP67** (impermeabile e resistente alla polvere), quindi è robusto anche in condizioni ambientali difficili.

📄 Basato su 3 chunk


In [7]:
# Test con domanda fuori dai documenti
domanda_off = "Quali sono i migliori smartphone del 2025?"
risposta_off, _ = chat_rag(domanda_off)
print(f"❓ {domanda_off}")
print(f"\n🤖 {risposta_off}")
print("\n💡 Il sistema dovrebbe rifiutarsi di rispondere!")

❓ Quali sono i migliori smartphone del 2025?

🤖 Non ho questa informazione nei miei documenti.

I documenti che ho a disposizione riguardano i prodotti e servizi di WiData Srl, un'azienda specializzata in IoT e smart cities. Non contengono informazioni su smartphone.

Se hai domande su sensori IoT, gateway, piani di abbonamento o servizi di WiData, sarò felice di aiutarti! 😊

💡 Il sistema dovrebbe rifiutarsi di rispondere!


---
## ⭐ Esercizi

In [8]:
NOME_STUDENTE = "Lorenzo"  # ← SCRIVI IL TUO NOME
if NOME_STUDENTE:
    print(f"✅ Notebook di: {NOME_STUDENTE}")
else:
    print("⚠️ Scrivi il tuo nome!")

✅ Notebook di: Lorenzo


### Esercizio 1 — Indicizza un documento tuo ★☆☆
Crea un documento di testo su un argomento a tua scelta (può essere anche una dispensa del corso, una ricetta, un regolamento). Indicizzalo in ChromaDB e fai 3 domande. I chunk recuperati sono rilevanti?

In [9]:
# ESERCIZIO 1
mio_documento = """
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in palestra.
I parametri più utili sono peso, massa muscolare, massa grassa e circonferenze.
Una misurazione coerente va fatta sempre nelle stesse condizioni: mattina, digiuno, stessa bilancia.
La costanza è più importante del singolo valore.
"""

mia_collection = chroma_client.get_or_create_collection(name="mio_doc_fitness")

chunks_mio = chunka_testo(mio_documento)
mia_collection.add(
    documents=chunks_mio,
    ids=[f"mio_chunk_{i}" for i in range(len(chunks_mio))]
)

def cerca_mio(domanda, n_risultati=3):
    risultati = mia_collection.query(
        query_texts=[domanda],
        n_results=n_risultati
    )
    return risultati["documents"][0]

domande_mie = [
    "Quali parametri devo monitorare?",
    "Quando è meglio fare la misurazione?",
    "Cosa conta di più nei progressi?"
]

for domanda in domande_mie:
    print(f"\n❓ {domanda}")
    for i, chunk in enumerate(cerca_mio(domanda, n_risultati=2)):
        print(f" 📄 Chunk {i+1}: {chunk[:120]}...")


❓ Quali parametri devo monitorare?
 📄 Chunk 1: 
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in pale...
 📄 Chunk 2: lore.
...

❓ Quando è meglio fare la misurazione?
 📄 Chunk 1: 
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in pale...
 📄 Chunk 2: lore.
...

❓ Cosa conta di più nei progressi?
 📄 Chunk 1: 
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in pale...
 📄 Chunk 2: lore.
...


### Esercizio 2 — Sperimenta con il chunking ★★☆
Prova a indicizzare lo stesso documento con chunk_size=200, 400 e 800. Per la stessa domanda, i chunk recuperati sono diversi? Quale dimensione dà risultati migliori?

In [10]:
# ESERCIZIO 2
domanda_test = "Inserisci qui una domanda sul tuo documento" # ← modifica

for chunk_size in [200, 400, 800]:
    print(f"\n{'='*50}")
    print(f"chunk_size = {chunk_size}")
    print('='*50)

    collection_tmp = chroma_client.get_or_create_collection(name=f"doc_{chunk_size}")
    chunks_tmp = chunka_testo(mio_documento, chunk_size=chunk_size)
    collection_tmp.add(
        documents=chunks_tmp,
        ids=[f"chunk_{i}_{chunk_size}" for i in range(len(chunks_tmp))]
    )

    risultati = collection_tmp.query(
        query_texts=[domanda_test],
        n_results=2
    )["documents"][0]

    for i, chunk in enumerate(risultati):
        print(f"📄 Risultato {i+1}: {chunk[:150]}...")

# Commento: quale chunk_size ha dato i risultati migliori?
# Risposta: ...


chunk_size = 200
📄 Risultato 1: ancia.
La costanza è più importante del singolo valore.
...
📄 Risultato 2: no peso, massa muscolare, massa grassa e circonferenze.
Una misurazione coerente va fatta sempre nelle stesse condizioni: mattina, digiuno, stessa bil...

chunk_size = 400
📄 Risultato 1: 
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in palestra.
I parametri più utili so...
📄 Risultato 2: lore.
...

chunk_size = 800
📄 Risultato 1: 
Guida rapida al fitness tracking.

Il monitoraggio della composizione corporea può aiutare a capire i progressi in palestra.
I parametri più utili so...


### Esercizio 3 — RAG + storia conversazione ★★☆
Integra RAG nella funzione `chat()` della Lezione 3 (quella con la history). Ogni risposta deve usare sia il contesto RAG che la storia della conversazione.

In [11]:
# ESERCIZIO 3
history = []

def chat_rag_con_storia(domanda):
    """Chatbot con RAG + conversazione multi-turno."""
    chunks_rilevanti = cerca(domanda, n_risultati=3)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    messaggio = f"""
Contesto dai documenti:
{contesto}

Domanda utente: {domanda}
"""

    history.append({"role": "user", "content": messaggio})

    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        system=SYSTEM_WIDATA,
        messages=history
    ).content[0].text

    history.append({"role": "assistant", "content": risposta})
    return risposta

# Test: domande collegate che richiedono sia RAG che memoria
print(chat_rag_con_storia("Parlami del sensore XS200"))
print()
print(chat_rag_con_storia("Qual è la sua autonomia?"))
print()
print(chat_rag_con_storia("Costa molto?"))

# Sensore XS200 - Monitoraggio Ambientale

Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

**Parametri misurati:**
- Temperatura: da -20°C a +60°C
- Umidità relativa: 0-100%
- Pressione atmosferica
- Qualità dell'aria (CO2, PM2.5)

**Caratteristiche tecniche:**
- Classificazione IP67: impermeabile e resistente alla polvere
- Alimentazione: batteria Li-Ion 3.7V
- Autonomia: (il documento non specifica la durata completa)

È uno strumento robusto e affidabile per applicazioni di monitoraggio ambientale in contesti industriali e urbani.

In base ai documenti forniti, l'autonomia del sensore XS200 non è completamente specificata. 

Il documento indica che il sensore è alimentato da una **batteria Li-Ion 3.7V** e menziona un'autonomia, ma il valore esatto non è riportato nel testo disponibile.

Per informazioni dettagliate sull'autonomia, ti consiglio di contattare il supporto tecnico:
- **Email:** support@widata.cloud
- **Telefono:** +39 079 

### Esercizio 4 — Chatbot RAG WiData completo ★★★ (Deliverable!)

Costruisci il chatbot completo con:
- RAG sul documento WiData
- Conversation history (sliding window)
- Streaming
- System prompt WiData con istruzione anti-hallucination
- Loop interattivo con `input()`
- Stampa i chunk usati per ogni risposta (per debug)

In [13]:
# ESERCIZIO 4 — Chatbot RAG completo (DELIVERABLE)
import chromadb, json, os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

SYSTEM = """
Sei l'assistente virtuale di WiData Srl.
Rispondi solo usando le informazioni presenti nei documenti forniti.
Se la risposta non è presente nei documenti, dì chiaramente che non hai quell'informazione.
Non inventare mai dati o dettagli.
Mantieni un tono professionale e chiaro.
"""

MAX_MESSAGGI = 10

def setup_rag(testo):
    """Indicizza il documento e restituisce la collection."""
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection(
        name="widata_docs_final",
        metadata={"hnsw:space": "cosine"}
    )
    chunks = chunka_testo(testo)
    collection.add(
        documents=chunks,
        ids=[f"chunk_{i}" for i in range(len(chunks))]
    )
    return collection

def cerca(domanda, collection, n=3):
    """Ricerca semantica nella collection."""
    risultati = collection.query(
        query_texts=[domanda],
        n_results=n
    )
    return risultati["documents"][0]

def chat_completo(domanda, history, collection):
    """Chatbot con RAG + storia + streaming."""
    chunks_rilevanti = cerca(domanda, collection, n=3)
    contesto = "\n\n---\n\n".join(chunks_rilevanti)

    prompt = f"""
Contesto dai documenti:
{contesto}

Domanda utente: {domanda}
"""

    history.append({"role": "user", "content": prompt})
    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]

    full_text = ""
    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        system=SYSTEM,
        messages=history
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            full_text += text

    print()
    history.append({"role": "assistant", "content": full_text})

    if len(history) > MAX_MESSAGGI:
        history[:] = history[-MAX_MESSAGGI:]

    print(f"📄 Chunk usati: {len(chunks_rilevanti)}")
    for i, c in enumerate(chunks_rilevanti):
        print(f"  - Chunk {i+1}: {c[:100]}...")
    return full_text

def main():
    collection = setup_rag(documento_widata)
    history = []
    print("🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.\n")

    while True:
        utente = input("Tu: ")
        if utente.lower() == "esci":
            print("👋 Arrivederci!")
            break
        print("WiData:", end=" ")
        chat_completo(utente, history, collection)

main() # Decommentare per eseguire

🤖 Chatbot WiData RAG avviato. Digita 'esci' per uscire.

Tu: Dimmi il tuo sensore migliore
WiData: # Il Nostro Sensore XS200

In base alla documentazione disponibile, posso descriverti il **sensore XS200**, che è progettato per il monitoraggio ambientale in ambienti industriali e urbani.

## Caratteristiche principali:

- **Misurazioni**: 
  - Temperatura (-20°C a +60°C)
  - Umidità relativa (0-100%)
  - Pressione atmosferica
  - Qualità dell'aria (CO2, PM2.5)

- **Robustezza**: Classificazione IP67 (impermeabile e resistente alla polvere)
- **Alimentazione**: Batteria Li-Ion 3.7V

Tuttavia, la documentazione disponibile non contiene informazioni complete per farmi dire quale sia il nostro sensore "migliore" in assoluto, poiché non ho dettagli su altri modelli eventualmente disponibili nel catalogo.

Se desideri conoscere meglio il sensore XS200 o scoprire altre soluzioni, ti consiglio di contattare il nostro team commerciale:
- **Email**: sales@widata.cloud
- **Telefono**: +39 079 123

---
## 📤 Consegna

1. Completa tutti gli esercizi
2. Scarica: `File → Scarica → .ipynb`
3. Rinomina: `Lezione4_TUONOME.ipynb`
4. Carica su GitHub in `lezione4/`

```bash
git add lezione4/
git commit -m "Lezione 4 completata"
git push
```

---
### 📖 Per la prossima lezione (Giovedì 04/06)
Leggi **Huyen Cap. 6 — sezione Agents**

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*